# Project 2 - Sequential workflow

We use a fictional job description and candidate profile to make a short cover letter. The first model call extracts job requirements. The second model call uses those requirements and the candidate profile to write the letter.

`@traceable` records an ordinary Python function as a run. When `create_cover_letter` calls the two traced steps, LangSmith shows one request trace with the steps and their model calls nested inside it. We attach metadata to the complete request so we can find it on the LangSmith dashboard.


In [1]:
from pathlib import Path
import sys

print("Working directory:", Path.cwd())
print("Python interpreter:", sys.executable)

Working directory: <repo>\2_sequential_workflow
Python interpreter: <repo>\.venv\Scripts\python.exe


In [2]:
job_description = Path("docs/dataset/job_description.txt").read_text(
    encoding="utf-8"
).strip()

print(job_description)

Junior Data Analyst — Cedar Health (fictional)

We need someone who can:
- Use SQL to answer questions from our data.
- Clean and analyze weekly CSV reports with Python.
- Summarize findings clearly for nontechnical teammates.

Experience with Power BI is helpful but not required.


In [3]:
candidate_profile = Path("docs/dataset/candidate_profile.txt").read_text(
    encoding="utf-8"
).strip()

print(candidate_profile)

Maya Rao — fictional candidate

- Used SQL to answer questions about a college club's membership data.
- Cleaned weekly CSV exports with Python and pandas for a class project.
- Presented short summaries of the findings to nontechnical classmates.
- Has not used Power BI.


### See each step's output

Run `extract_requirements` to see the intermediate list. Then pass that list and the candidate profile to `write_cover_letter`. These two notebook calls make separate traces; the combined request below shows both steps in one trace.


In [4]:
from tools.workflow import extract_requirements

requirements = extract_requirements(job_description)
print(requirements)

<repo>\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


- Use SQL to answer questions from our data.  
- Clean and analyze weekly CSV reports with Python.  
- Summarize findings clearly for nontechnical teammates.


In [5]:
from tools.workflow import write_cover_letter

letter = write_cover_letter(requirements, candidate_profile)
print(letter)

Dear Hiring Team,

I am excited to apply for the Junior Data Analyst position at Cedar Health. In my recent academic projects, I used SQL to extract insights from a college club’s membership database, cleaned and analyzed weekly CSV exports with Python and pandas, and presented concise, actionable summaries to nontechnical classmates—skills that directly align with your needs for SQL query development, Python-based data cleaning, and clear communication to diverse stakeholders.


### Trace the complete request

`create_cover_letter` is the parent run. Its `extract_requirements` and `write_cover_letter` calls are child runs, and each model call appears under the step that made it. The requirements from the first step become an input to the second step.

The `langsmith_extra` metadata labels this request with a case ID, prompt version, and lesson name. It helps us filter for this trace without changing either model prompt.


In [ ]:
from tools.workflow import create_cover_letter

result = create_cover_letter(
    job_description,
    candidate_profile,
    langsmith_extra={
        "metadata": {
            "case_id": "cedar-health-001",
            "prompt_version": "v1",
            "lesson": "sequential-workflow",
        }
    },
)

print("Requirements:\n", result["requirements"])
print("\nCover letter:\n", result["letter"])

### Inspect the trace in LangSmith

Open **sequential-workflow -> Local -> Tracing** and find the request with `case_id = cedar-health-001`. Expand the run tree:

```text
cover_letter_request
|-- extract_requirements
|   `-- model call
`-- write_cover_letter
    `-- model call
```

Inspect the first step's output and the second step's input to see the handoff. The two model calls run in order because the letter needs the extracted requirements. The metadata identifies this request; it is not part of the letter text.
